# Kultiva – Crop Recommendation AI | End-to-End ML Pipeline

This notebook implements the complete Kultiva machine learning pipeline for crop recommendation based on soil and environmental parameters.

## Dataset Features:
- **Nitrogen**: Nitrogen content in soil (kg/ha)
- **Phosphorus**: Phosphorus content in soil (kg/ha)
- **Potassium**: Potassium content in soil (kg/ha)
- **Temperature**: Temperature in Celsius
- **Humidity**: Relative humidity in %
- **pH_Value**: pH value of soil
- **Rainfall**: Rainfall in mm
- **Crop**: Target crop type (label)

## 1. Import Required Libraries

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier

# Model persistence
import joblib
import time

# Utils
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("All libraries imported successfully!")

## 2. Data Loading and Exploration

In [ ]:
def load_kultiva_data_safely(file_path):
    print("Loading Kultiva data... packing the backpack smartly!")

    # 1. We tell the computer exactly what size 'lunchboxes' to use
    memory_saving_dtypes = {
        'Nitrogen': np.int16,      # Small box for whole numbers
        'Phosphorus': np.int16,
        'Potassium': np.int16,
        'Temperature': np.float32, # Medium box for decimals
        'Humidity': np.float32,
        'pH_Value': np.float32,
        'Rainfall': np.float32,
        'Crop': 'category'         # Flashcards instead of heavy text!
    }

    # 2. Load the data using our strict memory rules
    df = pd.read_csv(file_path, dtype=memory_saving_dtypes)

    # Let's prove we saved memory!
    memory_used = df.memory_usage(deep=True).sum() / 1024 # in Kilobytes
    print(f"Data loaded successfully! Shape: {df.shape}")
    print(f"Total Memory Used: Only {memory_used:.2f} KB!")

    return df

# --- RUN IT ---
# Make sure the CSV file is in the same folder as your script
df_kultiva = load_kultiva_data_safely('Crop_Recommendation.csv')

In [ ]:
# Basic information about the dataset
print("Dataset Info:")
print(df_kultiva.info())

print("\nStatistical Summary:")
df_kultiva.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df_kultiva.isnull().sum())

# Check for duplicate rows
print(f"\nDuplicate rows: {df_kultiva.duplicated().sum()}")

# Check unique crops
print(f"\nNumber of unique crops: {df_kultiva['Crop'].nunique()}")
print("Crop types:", sorted(df_kultiva['Crop'].unique()))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of target variable
plt.figure(figsize=(12, 6))
sns.countplot(data=df_kultiva, y='Crop', order=df_kultiva['Crop'].value_counts().index)
plt.title('Distribution of Crop Types – Kultiva Dataset')
plt.xlabel('Count')
plt.ylabel('Crop Type')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions
features = ['Nitrogen', 'Phosphorus', 'Potassium', 'Temperature', 'Humidity', 'pH_Value', 'Rainfall']

plt.figure(figsize=(15, 10))
for i, feature in enumerate(features, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df_kultiva[feature].astype(float), kde=True)
    plt.title(f'Distribution of {feature}')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = df_kultiva.drop('Crop', axis=1).astype(float).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix – Kultiva')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for outlier detection
plt.figure(figsize=(15, 10))
for i, feature in enumerate(features, 1):
    plt.subplot(3, 3, i)
    sns.boxplot(y=df_kultiva[feature].astype(float))
    plt.title(f'Box Plot of {feature}')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Train-Test Split

In [ ]:
print("Splitting the flashcards into a Study Guide and a Practice Test...")

# 1. Separate the questions (X) from the answers (y)
# X = The weather and soil conditions
# y = The actual crop name
X = df_kultiva.drop(columns=['Crop'])
y = df_kultiva['Crop']

# 2. Hide 20% of the data for the final exam
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # This makes sure every crop type gets equal representation on the test!
)

print(f"Study Guide size: {X_train.shape[0]} flashcards")
print(f"Practice Test size: {X_test.shape[0]} flashcards")
print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 5. Model Training – Kultiva AI Brain (HistGradientBoostingClassifier)

In [ ]:
# 3. Meet your new, lightweight AI
# This model automatically uses the 'flashcard' binning technique
kultiva_model = HistGradientBoostingClassifier(
    max_iter=100,         # Maximum number of learning cycles
    learning_rate=0.1,    # How fast the model learns
    early_stopping=True,  # Crucial for laptops: it stops automatically if it stops getting smarter!
    random_state=42
)

# Time to study!
print("\nAI is studying... (Watch how fast this is!)")
start_time = time.time()

# We tell the AI to learn the patterns from the Study Guide
kultiva_model.fit(X_train, y_train)

end_time = time.time()
print(f"Done studying! It took only {end_time - start_time:.2f} seconds.")

# The Final Exam
accuracy = kultiva_model.score(X_test, y_test)
print(f"\nFinal Exam Score (Accuracy): {accuracy * 100:.2f}%")

## 6. Detailed Model Evaluation

In [ ]:
# Detailed classification report
y_pred = kultiva_model.predict(X_test)

print(f"Best Model: HistGradientBoostingClassifier (Kultiva Brain)")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
plt.figure(figsize=(14, 12))
crop_classes = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=crop_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=crop_classes,
            yticklabels=crop_classes)
plt.title('Confusion Matrix – Kultiva HistGradientBoostingClassifier')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=45, va='top')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation
print("Running 5-fold cross-validation...")
cv_scores = cross_val_score(kultiva_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Std Dev (±): {cv_scores.std() * 2:.4f}")

## 7. Save the Kultiva AI Brain

In [ ]:
print("Saving the AI's brain... hitting 'Save Game'!")

# 1. Choose a name for your saved game file
model_filename = 'kultiva_agri_brain.joblib'

# 2. Save the model to your hard drive
joblib.dump(kultiva_model, model_filename)
print(f"Success! Model securely saved as '{model_filename}'")

## 8. Inference – Testing the App Interface

In [ ]:
print("\n--- Testing the App Interface ---")
# 3. Load the brain back up (simulating what happens when a farmer opens your app)
loaded_kultiva_app = joblib.load(model_filename)

# 4. Create a fake farm profile (Let's pretend a farmer typed this into your app)
# Order: Nitrogen, Phosphorus, Potassium, Temperature, Humidity, pH, Rainfall
fake_farm_conditions = [[90, 42, 43, 20.8, 82.0, 6.5, 202.9]]

# 5. Ask the loaded brain for a recommendation
prediction = loaded_kultiva_app.predict(fake_farm_conditions)

print(f"Based on the soil and weather, Kultiva recommends planting: {prediction[0]}")

In [ ]:
# Test with multiple farm profiles (Batch Prediction)
print("\nTesting Batch Prediction:")
test_data = pd.DataFrame({
    'Nitrogen':    [90,    60,    40],
    'Phosphorus':  [42,    55,    35],
    'Potassium':   [43,    44,    40],
    'Temperature': [20.88, 23.00, 25.50],
    'Humidity':    [82.00, 80.50, 75.20],
    'pH_Value':    [6.50,  7.80,  6.20],
    'Rainfall':    [202.94, 263.96, 180.50]
})

batch_predictions = loaded_kultiva_app.predict(test_data)
test_data['Recommended_Crop'] = batch_predictions
print("Batch Prediction Results:")
test_data

In [ ]:
# Test against actual test set
print("\nTesting with Actual Test Data (first 5 samples):")
test_sample = X_test.iloc[0:5].copy()
sample_preds = loaded_kultiva_app.predict(test_sample)

test_sample = test_sample.copy()
test_sample['Predicted_Crop'] = sample_preds
test_sample['Actual_Crop'] = y_test.iloc[0:5].values
test_sample['Correct'] = test_sample['Predicted_Crop'] == test_sample['Actual_Crop']

print("Test Data Predictions:")
test_sample[['Nitrogen','Phosphorus','Potassium','Temperature','Humidity','pH_Value','Rainfall',
             'Predicted_Crop','Actual_Crop','Correct']]

## 9. Model Performance Summary

In [ ]:
print("=" * 50)
print("     KULTIVA CROP RECOMMENDATION AI SUMMARY")
print("=" * 50)
print(f"Model Type       : HistGradientBoostingClassifier")
print(f"Test Accuracy    : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"CV Score (mean)  : {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print(f"Number of Features: {len(X.columns)}")
print(f"Number of Crop Classes: {df_kultiva['Crop'].nunique()}")
print(f"Training Samples : {X_train.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")

print("\nFeatures Used:")
for i, feature in enumerate(X.columns, 1):
    print(f"  {i}. {feature}")

print("\nCrop Classes:")
for i, crop in enumerate(sorted(df_kultiva['Crop'].unique()), 1):
    print(f"  {i}. {crop}")

print(f"\nModel saved as   : {model_filename}")

print("\n" + "=" * 50)
print("   INFERENCE TEST")
print("=" * 50)
print(f"Single prediction test  : {prediction[0]}")
sample_acc = test_sample['Correct'].mean()
print(f"Batch accuracy on sample: {sample_acc:.4f}")
print("\nKultiva AI is ready for deployment!")

## 10. Usage Instructions

### For Single Prediction:
```python
import joblib

# Load the saved Kultiva brain
model = joblib.load('kultiva_agri_brain.joblib')

# Order: Nitrogen, Phosphorus, Potassium, Temperature, Humidity, pH_Value, Rainfall
result = model.predict([[90, 42, 43, 20.88, 82.00, 6.50, 202.94]])
print(f"Recommended crop: {result[0]}")
```

### For Batch Prediction:
```python
import pandas as pd

data = pd.DataFrame({
    'Nitrogen': [...], 'Phosphorus': [...], 'Potassium': [...],
    'Temperature': [...], 'Humidity': [...], 'pH_Value': [...], 'Rainfall': [...]
})
predictions = model.predict(data)
```

### Model File Required:
- `kultiva_agri_brain.joblib` — Trained Kultiva model (only file needed)